# 03 - Seleccion del mejor modelo

Lee las corridas historicas guardadas por `02_model_training.ipynb`, elige la mejor por `best_map` y la evalua en test.

## 1. Imports y configuracion

In [ ]:
# Importa dependencias y helpers para seleccionar y evaluar la mejor corrida guardada.
from pathlib import Path
import json
import shutil
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from prod.detection_dataset import (
    CarDamageDetectionDataset,
    ComposeDetection,
    ToTensorDetection,
    collate_fn,
)
from prod.detection_metrics import evaluate_map
from prod.detection_models import create_model_from_config
from prod.detection_training import load_checkpoint
from utils import (
    export_results_comparison_html,
    load_experiment_runs,
    resolve_portable_path,
    to_portable_path,
)


In [ ]:
# Define rutas, dispositivo y parámetros mínimos para la evaluación final.
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "dev" / "experiments"
RUNS_MANIFEST_PATH = PROJECT_ROOT / "dev" / "runs_manifest.jsonl"
RESULTS_HTML_PATH = PROJECT_ROOT / "dev" / "results_comparison.html"
BEST_TEST_RESULT_PATH = PROJECT_ROOT / "dev" / "best_test_result.json"
FINAL_MODEL_PATH = PROJECT_ROOT / "dev" / "modelo.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 5
NUM_WORKERS = 0

print("Device:", DEVICE)
print("RUNS_MANIFEST_PATH:", RUNS_MANIFEST_PATH)


## 2. Cargar corridas acumuladas

In [ ]:
# Carga el manifest histórico, ordena corridas por mAP y vuelve a exportar el reporte HTML.
results_df = load_experiment_runs(RUNS_MANIFEST_PATH)
if results_df.empty:
    raise ValueError(f"No hay corridas guardadas en {RUNS_MANIFEST_PATH}. Primero corre 02_model_training.ipynb.")

results_df = results_df.sort_values(by="best_map", ascending=False).reset_index(drop=True)
results_html_path = export_results_comparison_html(
    results_df.drop(columns=["config", "history"], errors="ignore"),
    RESULTS_HTML_PATH,
)
print(f"Tabla acumulada exportada a HTML en: {results_html_path}")
results_df.drop(columns=["config", "history"], errors="ignore")


## 3. Elegir mejor corrida

In [ ]:
# Recupera la mejor corrida y resuelve la ruta portable del checkpoint elegido.
best_run = results_df.iloc[0].to_dict()
best_config = best_run["config"]
best_checkpoint_path = resolve_portable_path(
    best_run["checkpoint_path"],
    base_dir=PROJECT_ROOT,
    fallback_dir=ARTIFACTS_DIR,
)

print(json.dumps({
    "run_id": best_run["run_id"],
    "name": best_run["name"],
    "best_epoch": best_run["best_epoch"],
    "best_map": best_run["best_map"],
    "best_map_50": best_run["best_map_50"],
    "checkpoint_path": to_portable_path(best_checkpoint_path, base_dir=PROJECT_ROOT),
}, indent=2))


## 4. Reconstruir DataLoader de test

In [ ]:
# Construye el dataset y dataloader de test compatibles con la configuración ganadora.
eval_transform = ComposeDetection([
    ToTensorDetection(),
])

test_dataset = CarDamageDetectionDataset(
    data_dir=DATA_DIR,
    split="test",
    transform=eval_transform,
    model_name=best_config["model_name"],
    resize=best_config["resize"],
    image_size=best_config["image_size"],
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

print("Test samples:", len(test_dataset))


## 5. Evaluacion final en test

In [ ]:
# Carga el checkpoint ganador, evalúa mAP en test y guarda el resultado final.
best_model = create_model_from_config(best_config).to(DEVICE)
_ = load_checkpoint(best_model, best_checkpoint_path, device=DEVICE)

test_metrics = evaluate_map(
    model=best_model,
    dataloader=test_loader,
    device=DEVICE,
    class_metrics=True,
)

best_test_result = {
    "run_id": best_run["run_id"],
    "best_experiment": best_run["name"],
    "checkpoint_path": to_portable_path(best_checkpoint_path, base_dir=PROJECT_ROOT),
    "test_map": test_metrics.get("map"),
    "test_map_50": test_metrics.get("map_50"),
    "test_map_75": test_metrics.get("map_75"),
    "test_mar_100": test_metrics.get("mar_100"),
}

BEST_TEST_RESULT_PATH.write_text(json.dumps(best_test_result, indent=2), encoding="utf-8")
print(json.dumps(best_test_result, indent=2))
print("Resultado final guardado en:", BEST_TEST_RESULT_PATH)


## 6. Guardado del modelo final

In [ ]:
# Copia el mejor checkpoint a `dev/modelo.pth` como modelo final promovido.
shutil.copy2(best_checkpoint_path, FINAL_MODEL_PATH)
print("Modelo final guardado en:", FINAL_MODEL_PATH)
